# Vérification de l'import CSV et affichage de la bbox

Ce notebook vérifie que `core.config.load_entities` lit correctement le CSV d'entités (`data/configuration/entites_exemple.csv`) et affiche l'emprise (bbox) résultante sur une carte `leafmap`.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities

## 1. Chargement du CSV via `load_entities`

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entities = load_entities(csv_path)
entities

## 2. Construction des géométries de bbox (EPSG:2154)

In [ ]:
import geopandas as gpd
from shapely.geometry import box

records = [
    {
        "nom_entite": e.nom,
        "type_entite": e.type_entite,
        "code_insee": e.code_insee,
        "geometry": box(*e.bbox),
    }
    for e in entities
]
gdf = gpd.GeoDataFrame(records, crs="EPSG:2154")
gdf

## 3. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 (attendu par leafmap) uniquement pour l'affichage — le traitement interne reste en EPSG:2154.

Backend `folium` (HTML statique) plutôt que `ipyleaflet` : plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

gdf_wgs84 = gdf.to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(gdf_wgs84, layer_name="Emprises", info_mode="on_click")
m.zoom_to_gdf(gdf_wgs84)
m